# Detección de Grooming Utilizando el Algoritmo Naive Bayes — Versión mejorada

**Materia:** Aprendizaje Automático  
**Docente:** Scudero, Yanina Ximena  
**Alumno:** Lucas Nahuel Llanes  
**Carrera:** Tecnicatura en Ciencia de Datos e Inteligencia Artificial  
**Institución:** INSTITUTO SUPERIOR DE FORMACIÓN TÉCNICA N° 197  
**Año:** 2026

Esta versión incorpora stopwords en español y n-gramas (1,2) al vectorizador.

## Importar Bibliotecas necesarias

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import nltk
    from nltk.corpus import stopwords
    try:
        stopwords.words('spanish')
    except LookupError:
        nltk.download('stopwords')
    spanish_stop = list(stopwords.words('spanish'))
except Exception:
    spanish_stop = ["y","de","la","el","que","en","a","los","las","un","una","por","para","con","no","se","del","al","lo","como","más","mi","tu","su","si","pero","o","ha","le"]

vectorizer_ng = CountVectorizer(
    ngram_range=(1,2),
    stop_words=spanish_stop,
    lowercase=True,
    token_pattern=r"(?u)\b\w\w+\b"
)

## Cargar los datos y crear el DataFrame

In [ ]:
data = {
    'text': [
        "¿Cuántos años tenés? No le digas a nadie que hablamos.",
        "Hola, ¿cómo estás? ¿Querés jugar algo online?",
        "Sos muy linda, no le muestres esto a tus papás.",
        "¿Terminaste la tarea de matemáticas?",
        "Podes mandarme una foto tuya, pero que sea solo para mí.",
        "¿Querés que te ayude con el juego que no podías pasar?",
        "No le contés a nadie que hablamos por acá, ¿sí?",
        "¿Te gustó la película de ayer?",
        "¿Estás solo en casa ahora?",
        "¿Querés hacer una videollamada para estudiar juntos?",
        "No le digas a tus padres que te escribí, es nuestro secreto.",
        "¿Cómo te fue en el examen de historia?",
        "Mandame una foto tuya, pero que nadie más la vea.",
        "¿Jugamos Minecraft esta tarde?",
        "Sos muy especial para mí, no le cuentes a nadie que hablamos.",
        "¿Querés que te pase los apuntes de biología?",
        "¿Podes mostrarme cómo estás vestida ahora?",
        "¿Tenés ganas de salir a andar en bici mañana?",
        "No hace falta que le digas a nadie que hablamos tanto.",
        "¿Querés que estudiemos juntos para el parcial?"
    ],
    'label': [
        "grooming", "no grooming", "grooming", "no grooming", "grooming",
        "no grooming", "grooming", "no grooming", "grooming", "no grooming",
        "grooming", "no grooming", "grooming", "no grooming", "grooming",
        "no grooming", "grooming", "no grooming", "grooming", "no grooming"
    ]
}

df = pd.DataFrame(data)
X = df['text']
y = df['label']
print(df)

## Validación Cruzada con n-gramas y stopwords

In [ ]:
pipeline = Pipeline([
    ('vectorizer', vectorizer_ng),
    ('classifier', MultinomialNB())
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
print("Precisión en cada fold:", scores)
print("Precisión media:", scores.mean())
print("Desviación estándar:", scores.std())

## Entrenamiento del modelo

In [ ]:
pipeline.fit(X, y)
model = pipeline.named_steps['classifier']
vectorizer = pipeline.named_steps['vectorizer']

## Matriz de confusión

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

pipeline_split = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2), stop_words=spanish_stop, lowercase=True, token_pattern=r"(?u)\b\w\w+\b")),
    ('classifier', MultinomialNB())
])
pipeline_split.fit(X_train, y_train)
y_pred = pipeline_split.predict(X_test)
print("Reporte de clasificación en conjunto de prueba:\n", classification_report(y_test, y_pred))
print("Precisión en conjunto de prueba:", accuracy_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=pipeline_split.named_steps['classifier'].classes_)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=pipeline_split.named_steps['classifier'].classes_, yticklabels=pipeline_split.named_steps['classifier'].classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('confusion_matrix.png')
plt.show()

## Clasificación de nuevos mensajes

In [ ]:
nuevas_noticias = [
    "¿Podés mandarme una foto tuya? No se la muestres a nadie.",
    "¿Jugamos Minecraft esta tarde?",
    "No le digas a tus papás que hablamos, ¿sí?",
    "¿Terminaste el trabajo práctico de historia?"
]

predicciones = pipeline.predict(nuevas_noticias)

df_preds = pd.DataFrame({'texto': nuevas_noticias, 'prediccion': predicciones})
df_preds.style.set_properties(**{'text-align': 'left'})
df_preds